# Day 4 — Build Your First AI Agent

Today we move from a basic LLM app to an **AI agent**.

A chatbot usually answers in one shot.  
An agent can:
- decide what to do,
- use a tool,
- observe the result,
- and continue until it can answer.

In this notebook, we will build a **minimal tool-using agent** step by step.

## Learning goals

By the end of this notebook, you should be able to:

- explain what makes an AI agent different from a normal chatbot,
- understand the basic agent loop,
- define simple tools in Python,
- build a small agent that chooses and uses a tool,
- trace how the loop works internally.

This notebook covers the **first half** of Day 4.
Tomorrow you can extend it with memory, more tools, and better error handling.

## What is an AI agent?

A simple way to think about it:

**LLM app**
- user asks a question,
- model answers once.

**AI agent**
- user asks a question,
- model decides whether it needs a tool,
- tool runs,
- result comes back,
- model continues,
- final answer is returned.

That repeated cycle is the core idea behind many agent systems.

## The agent loop

A minimal agent often follows this pattern:

1. Read the user request.
2. Decide whether a tool is needed.
3. Call the tool.
4. Observe the result.
5. Repeat if needed.
6. Return the final answer.

Pseudo-flow:

User -> Model -> Tool -> Model -> Final Answer

In [1]:
# Basic imports for today's notebook

import math
import json
from typing import Callable, Dict

## Step 1 — Create tools

Tools are just functions the agent can use.

We will start with two simple tools:
- a calculator,
- a word counter.

These are intentionally small so the agent logic stays easy to understand.

In [2]:
def calculator(expression: str) -> str:
    """
    Evaluate a simple math expression safely.
    Example: '12 * (4 + 3)'
    """
    allowed_names = {
        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
        "pow": pow,
        "sqrt": math.sqrt,
    }

    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculator error: {e}"


def word_counter(text: str) -> str:
    """
    Count the number of words in a string.
    """
    words = text.split()
    return f"Word count: {len(words)}"

In [3]:
tools: Dict[str, Callable[[str], str]] = {
    "calculator": calculator,
    "word_counter": word_counter,
}

tools

{'calculator': <function __main__.calculator(expression: str) -> str>,
 'word_counter': <function __main__.word_counter(text: str) -> str>}

## Step 2 — Test the tools directly

Before letting an agent use tools, always test the tools on their own.

In [4]:
print(calculator("12 * (4 + 3)"))
print(word_counter("AI agents can reason and use tools"))

Result: 84
Word count: 7


## Step 3 — Build a tiny rule-based agent

Real agents often use an LLM to decide which tool to call.

For learning, we will first build a very small **rule-based agent**.
This helps us understand the loop before adding a real model.

In [5]:
def choose_tool(user_query: str):
    query = user_query.lower()

    if any(word in query for word in ["calculate", "math", "+", "-", "*", "/", "sum"]):
        return "calculator"

    if any(word in query for word in ["count words", "word count", "how many words"]):
        return "word_counter"

    return None

## Step 4 — Build the agent loop

This is the key idea.

The agent will:
- inspect the request,
- choose a tool if needed,
- run the tool,
- and then produce a final response.

This is a simplified version of the agent loop used in larger frameworks.

In [6]:
def run_agent(user_query: str) -> str:
    print(f"User: {user_query}")

    tool_name = choose_tool(user_query)

    if tool_name is None:
        return "I do not need a tool here. I can answer directly or ask for clarification."

    print(f"Agent decided to use tool: {tool_name}")

    if tool_name == "calculator":
        expression = (
            user_query.lower()
            .replace("calculate", "")
            .replace("math", "")
            .strip()
        )
        tool_result = tools[tool_name](expression)

    elif tool_name == "word_counter":
        text = (
            user_query.lower()
            .replace("count words in", "")
            .replace("word count for", "")
            .replace("how many words in", "")
            .strip()
        )
        tool_result = tools[tool_name](text)

    else:
        tool_result = "Unknown tool."

    print(f"Tool output: {tool_result}")

    final_answer = f"Final answer: {tool_result}"
    return final_answer

In [7]:
response = run_agent("calculate 25 * (2 + 6)")
print(response)

User: calculate 25 * (2 + 6)
Agent decided to use tool: calculator
Tool output: Result: 200
Final answer: Result: 200


In [8]:
response = run_agent("count words in AI agents are useful for multi-step tasks")
print(response)

User: count words in AI agents are useful for multi-step tasks
Agent decided to use tool: calculator
Tool output: Calculator error: invalid syntax (<string>, line 1)
Final answer: Calculator error: invalid syntax (<string>, line 1)


## What just happened?

Our mini-agent followed this pattern:

- received a user request,
- selected a tool,
- passed input to the tool,
- received the tool output,
- turned that into a final answer.

That is the basic structure behind many tool-using agents.

## Step 5 — Add a traceable loop

Many tutorials teach agents as a visible loop because it makes debugging easier.

We will now make the steps explicit:
- Thought
- Action
- Observation
- Final Answer

This is inspired by the common ReAct-style pattern.

In [9]:
def run_agent_with_trace(user_query: str) -> str:
    print("=== AGENT TRACE START ===")
    print(f"User Query: {user_query}")

    tool_name = choose_tool(user_query)

    if tool_name is None:
        print("Thought: I do not need a tool.")
        print("Final Answer: I can answer directly or ask for clarification.")
        print("=== AGENT TRACE END ===")
        return "I can answer directly or ask for clarification."

    print(f"Thought: I should use the {tool_name} tool.")

    if tool_name == "calculator":
        expression = (
            user_query.lower()
            .replace("calculate", "")
            .replace("math", "")
            .strip()
        )
        print(f"Action: calculator('{expression}')")
        observation = calculator(expression)

    elif tool_name == "word_counter":
        text = (
            user_query.lower()
            .replace("count words in", "")
            .replace("word count for", "")
            .replace("how many words in", "")
            .strip()
        )
        print(f"Action: word_counter('{text}')")
        observation = word_counter(text)

    else:
        observation = "Unknown tool."

    print(f"Observation: {observation}")
    final_answer = f"Based on the tool result, the answer is: {observation}"
    print(f"Final Answer: {final_answer}")
    print("=== AGENT TRACE END ===")

    return final_answer

In [10]:
run_agent_with_trace("calculate 144 / 12")

=== AGENT TRACE START ===
User Query: calculate 144 / 12
Thought: I should use the calculator tool.
Action: calculator('144 / 12')
Observation: Result: 12.0
Final Answer: Based on the tool result, the answer is: Result: 12.0
=== AGENT TRACE END ===


'Based on the tool result, the answer is: Result: 12.0'

In [11]:
run_agent_with_trace("how many words in agents use tools to finish tasks")

=== AGENT TRACE START ===
User Query: how many words in agents use tools to finish tasks
Thought: I should use the word_counter tool.
Action: word_counter('agents use tools to finish tasks')
Observation: Word count: 6
Final Answer: Based on the tool result, the answer is: Word count: 6
=== AGENT TRACE END ===


'Based on the tool result, the answer is: Word count: 6'

## Limitations of this version

This is a good teaching version, but it is still very limited.

Current limitations:
- tool selection is rule-based, not model-based,
- input parsing is fragile,
- there is no memory,
- there is no retry or error recovery,
- the agent can only do one simple step at a time.

Tomorrow, you can improve this by adding:
- more tools,
- better parsing,
- short-term memory,
- a real LLM-based decision step.

## Mini exercise

Try one or more of these:

1. Add a new tool called `character_counter`.
2. Update `choose_tool()` so the agent can select it.
3. Test the agent with your own examples.
4. Make the final answer sound more natural.

Suggested challenge:
- Can you make the agent support both word count and character count?

In [ ]:
# Your practice space

# Example:
# def character_counter(text: str) -> str:
#     return f"Character count: {len(text)}"

pass

## Recap

Today you built:
- simple Python tools,
- a tool-selection function,
- a minimal agent,
- and a traceable agent loop.

That is the foundation for more advanced agent systems.
Next, you can add memory, multiple tools, and a real LLM to make the agent more flexible.

## Push-ready note

This is a solid **50% Day 4 notebook** because it already teaches the core agent idea:
an agent is not just a model answer, but a loop that can decide, act, observe, and return a result.